# Tahap 1 — Membangun Case Base

Tahap ini mencakup:
1. **Scraping** putusan dari Direktori Mahkamah Agung RI
2. **Konversi PDF ke Plain Text** menggunakan pdfminer
3. **Cleaning & Normalisasi** teks putusan

**Input**: PDF putusan dari `data/raw/pasal_362/` dan `data/raw/pasal_363/`

**Output**: Teks bersih di `data/processed/clean/pasal_362/` dan `data/processed/clean/pasal_363/`

## 1.1 Import Library

In [ ]:
import re
import os
import time
import json
import random
import logging
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from pdfminer.high_level import extract_text
from pdfminer.pdfparser import PDFSyntaxError

## 1.2 Konfigurasi

In [ ]:
BASE_URL   = "https://putusan3.mahkamahagung.go.id"
PENGADILAN = "pn-malang"
KATEGORI   = "pencurian-1"

TARGET = {
    "pasal_362": 30,
    "pasal_363": 30,
}

RAW_DIR   = Path("../data/raw")
CLEAN_DIR = Path("../data/processed/clean")
LOG_DIR   = Path("../logs")
MIN_CHARS = 500

for d in [RAW_DIR, CLEAN_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "cleaning.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

## 1.3 Scraping Putusan

Mengunduh 60 dokumen putusan (30 Pasal 362 + 30 Pasal 363) dari Direktori Putusan Mahkamah Agung RI menggunakan `requests` dan `BeautifulSoup`.

> **Catatan**: Bagian ini memerlukan `cf_clearance` cookie yang valid. Jika cookie sudah kadaluarsa, perbarui nilainya dari browser.

In [ ]:
CF_CLEARANCE = ""
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
DELAY_MIN = 2.0
DELAY_MAX = 4.0

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
    "Connection": "keep-alive",
    "Referer": BASE_URL + "/",
    "Cookie": f"cf_clearance={CF_CLEARANCE}",
})


def delay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))


def get_listing_page(page):
    url = f"{BASE_URL}/direktori/index/pengadilan/{PENGADILAN}/kategori/{KATEGORI}/page/{page}.html"
    try:
        r = session.get(url, timeout=20)
        if r.status_code == 200:
            return BeautifulSoup(r.text, "lxml")
    except Exception as e:
        log.error(f"Gagal akses listing halaman {page}: {e}")
    return None


def parse_listing(soup):
    links = []
    rows = soup.select("table tbody tr")
    for row in rows:
        a_tag = row.select_one("td a[href]")
        if a_tag:
            href = a_tag.get("href", "")
            title = a_tag.get_text(strip=True)
            if href:
                full_url = href if href.startswith("http") else BASE_URL + href
                links.append({"url": full_url, "title": title})
    return links


def classify_pasal(text):
    text_lower = text.lower()
    if "363" in text_lower:
        return "pasal_363"
    elif "362" in text_lower:
        return "pasal_362"
    return None


def download_pdf(detail_url, output_path):
    try:
        r = session.get(detail_url, timeout=20)
        if r.status_code != 200:
            return False
        soup = BeautifulSoup(r.text, "lxml")
        pdf_link = soup.select_one("a[href$='.pdf']")
        if not pdf_link:
            return False
        pdf_url = pdf_link["href"]
        if not pdf_url.startswith("http"):
            pdf_url = BASE_URL + pdf_url
        delay()
        r2 = session.get(pdf_url, timeout=30)
        if r2.status_code == 200 and len(r2.content) > 1000:
            output_path.write_bytes(r2.content)
            return True
    except Exception as e:
        log.error(f"Download gagal: {e}")
    return False


print("Scraper siap. Jalankan cell berikutnya untuk mulai scraping.")
print(f"Target: {TARGET}")

In [ ]:
collected = {k: 0 for k in TARGET}
page = 1
max_pages = 50

while any(collected[k] < TARGET[k] for k in TARGET) and page <= max_pages:
    soup = get_listing_page(page)
    if not soup:
        break
    links = parse_listing(soup)
    if not links:
        break
    
    for link in links:
        pasal = classify_pasal(link["title"])
        if pasal is None or collected.get(pasal, 0) >= TARGET.get(pasal, 0):
            continue
        
        out_dir = RAW_DIR / pasal
        out_dir.mkdir(parents=True, exist_ok=True)
        idx = collected[pasal] + 1
        filename = f"case_{pasal}_{idx:03d}.pdf"
        out_path = out_dir / filename
        
        if out_path.exists():
            collected[pasal] += 1
            continue
        
        delay()
        ok = download_pdf(link["url"], out_path)
        if ok:
            collected[pasal] += 1
            log.info(f"[{pasal}] {idx}/{TARGET[pasal]}: {filename}")
        else:
            log.warning(f"Gagal download: {link['title']}")
    
    page += 1
    delay()

print(f"\nHasil scraping: {collected}")

## 1.4 Konversi PDF ke Plain Text

Mengubah file PDF menjadi plain text menggunakan `pdfminer.six`. File teks disimpan di folder yang sama dengan file PDF.

In [ ]:
def pdf_to_text(pdf_path):
    try:
        text = extract_text(str(pdf_path))
        if not text or len(text.strip()) < MIN_CHARS:
            log.warning(f"Teks terlalu pendek ({len(text.strip() if text else '')} char): {pdf_path.name}")
            return None
        return text
    except PDFSyntaxError as e:
        log.error(f"PDF corrupt: {pdf_path.name} — {e}")
        return None
    except Exception as e:
        log.error(f"Error: {pdf_path.name}: {e}")
        return None


pdf_files = sorted(RAW_DIR.rglob("*.pdf"))
print(f"Total PDF ditemukan: {len(pdf_files)}")

success = failed = skipped = 0

for pdf_path in tqdm(pdf_files, desc="Konversi PDF"):
    txt_path = pdf_path.with_suffix(".txt")
    if txt_path.exists():
        skipped += 1
        continue
    text = pdf_to_text(pdf_path)
    if text is None:
        failed += 1
        continue
    txt_path.write_text(text, encoding="utf-8")
    success += 1

print(f"\nBerhasil: {success} | Gagal: {failed} | Skip: {skipped}")

## 1.5 Cleaning & Normalisasi Teks

Pipeline cleaning:
1. Hapus header/footer berulang (watermark MA, nomor halaman)
2. Hapus karakter non-latin
3. Normalisasi spasi dan baris kosong
4. Lowercase
5. Validasi minimal 100 kata

In [ ]:
def clean_text(text):
    patterns_hapus = [
        r"Direktori Putusan Mahkamah Agung Republik Indonesia",
        r"putusan\.mahkamahagung\.go\.id",
        r"Disclaimer\s*[::].*",
        r"kepaniteraan mahkamah agung.*",
        r"mahkamah agung republik indonesia",
        r"halaman \d+ dari \d+",
        r"hal\.\s*\d+\s*dari\s*\d+",
        r"-{3,}",
        r"={3,}",
        r"\f",
    ]
    for pat in patterns_hapus:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
    text = text.replace("\xa0", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"[^\x00-\x7F\u00C0-\u024F\u1E00-\u1EFF]", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"^\s+|\s+$", "", text)
    text = text.lower()
    return text


def validate_text(text, filename):
    word_count = len(text.split())
    if word_count < 100:
        log.warning(f"VALIDASI GAGAL — terlalu pendek ({word_count} kata): {filename}")
        return False
    return True


txt_files = sorted(RAW_DIR.rglob("*.txt"))
print(f"Total TXT ditemukan: {len(txt_files)}")

success = failed = skipped = 0

for txt_path in tqdm(txt_files, desc="Cleaning"):
    label_folder = txt_path.parent.name
    out_dir = CLEAN_DIR / label_folder
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / txt_path.name

    if out_path.exists():
        skipped += 1
        continue

    try:
        raw_text = txt_path.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        log.error(f"Gagal baca {txt_path.name}: {e}")
        failed += 1
        continue

    clean = clean_text(raw_text)

    if not validate_text(clean, txt_path.name):
        failed += 1
        continue

    out_path.write_text(clean, encoding="utf-8")
    success += 1

print(f"\nBerhasil: {success} | Gagal: {failed} | Skip: {skipped}")
print(f"Output: {CLEAN_DIR.resolve()}")

## 1.6 Validasi Hasil

Periksa jumlah file yang berhasil diproses dan tampilkan sample teks.

In [ ]:
clean_files = sorted(CLEAN_DIR.rglob("*.txt"))
print(f"Total file clean: {len(clean_files)}")

for folder in ["pasal_362", "pasal_363"]:
    count = len(list((CLEAN_DIR / folder).glob("*.txt")))
    print(f"  {folder}: {count} file")

if clean_files:
    sample = clean_files[0]
    text = sample.read_text(encoding="utf-8")
    print(f"\nSample ({sample.name}):")
    print(text[:500])
    print(f"\n... ({len(text.split())} kata total)")